In [1]:
# 📦 Installa le dipendenze (eseguito solo la prima volta)
!pip install -q transformers huggingface_hub accelerate

# 📚 Import
import os
import pandas as pd
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.8 MB/s eta 0:00:00:00:0100:01


2025-07-15 11:16:18.473000: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752578178.824377      70 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752578178.924402      70 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
# 🔐 Login Hugging Face
HUGGINGFACE_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", "HUGGINGFACE_TOKEN")
login(token=HUGGINGFACE_TOKEN)


"""
from huggingface_hub import HfApi

api = HfApi()
access_models = api.list_models()

for m in access_models:
    print(m.modelId)
"""

print("Login succesful")

Login succesful


In [3]:
# ⚙️ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")


✅ Device: cuda


In [4]:
# 📁 Configurazione
USE_SHORT_JOKES = True  # True = usa short-jokes, False = usa jester
USE_SAMPLE = True        # True = sample da 100 battute, False = intero dataset

# 📁 Etichette per i file
dataset_label = "short" if USE_SHORT_JOKES else "jester"
sample_label = "sample" if USE_SAMPLE else "full"

if USE_SHORT_JOKES:
    path = "/kaggle/input/short-jokes-dataset/train.csv"
    df = pd.read_csv(path)

    # Rinomina colonna per compatibilità
    if "text" in df.columns:
        df.rename(columns={"text": "jokeText"}, inplace=True)

else:
    path = "/kaggle/input/rated-jokes-dataset-from-jester/jester_clean_normalized_aggregated.csv"
    df = pd.read_csv(path)


# 📦 Sample o intero dataset
if USE_SAMPLE:
    df = df.sample(n=1000, random_state=42).reset_index(drop=True)
    print("✅ Sample da 100 battute selezionato")
else:
    df = df.reset_index(drop=True)
    print(f"✅ Usato l'intero dataset: {len(df)} battute")


✅ Sample da 100 battute selezionato


In [5]:
print(df.head(n=5))

                                            jokeText
0  What do all battered women have in common? The...
1  Who invented the North America? TEACHER: Sarah...
2  I feel like this election ended up being a goo...
3  What do you call a pile of kittens? A Meowntain\n
4  I feel bad for people named John Smith. They p...


In [6]:
# 🏷️ Humor categories con definizione
categories = {
    "Edgy Content": "Sex, taboos, death, shock",
    "Cultural Reference": "Politics, celebrities, pop culture",
    "Wordplay": "Puns, linguistic ambiguity",
    "Absurdity": "Illogical logic, surrealism, nonsense",
    "Relatable": "Everyday life, technology, relationships, self-irony",
    "Offensive Humor": "Insults, offensive, stereotypes, racism, sarcasm"
}


In [7]:
# 📦 Caricamento modello
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Padding per il batching
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )
    print(f"📍 Model loaded on: {model.device}")
    
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer) #, device=0)
    return pipe

# 🤖 Classify jokes with scores in structured JSON
def classify_jokes_with_scores(df, pipe, model_key, batch_size=32):
    prompts = []
    for joke in df["jokeText"]:
        category_list = "\n".join([f'- "{cat}": {desc}' for cat, desc in categories.items()])
        prompt = f"""You are a humor expert and researcher.
    
        For the following joke, rate from 0 to 10 how much it fits each of these categories based on their definitions.
        
        Categories:
        {category_list}
        
        Joke:
        \"\"\"{joke}\"\"\"
        
        Respond ONLY in this exact JSON format:
        {{
          "Edgy Content": 2,
          "Cultural Reference": 8,
          "Wordplay": 5,
          "Absurdity": 0,
          "Relatable": 3,
          "Insult Humor": 10 
        }}"""
        prompts.append(prompt)

    # Initialize score columns
    for cat in categories:
        df[f"{model_key}_{cat}"] = None

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        print(f"🌀 [{model_key}] Batch {i // batch_size + 1}")

        outputs = pipe(
            batch,
            max_new_tokens=500,
            do_sample=False,
            batch_size=batch_size,
            temperature=0.0
        )

        for j, output in enumerate(outputs):
            text = output[0]["generated_text"]
            try:
                json_start = text.index("{")
                json_text = text[json_start:].strip()
                scores = json.loads(json_text)
                for cat in categories:
                    df.at[i + j, f"{model_key}_{cat}"] = scores.get(cat, 0)
            except Exception as e:
                print(f"⚠️ JSON parsing error in batch {i // batch_size + 1}, item {j}: {e}")
                print(f"📝 Model output:\n{text}\n{'-'*60}")
                for cat in categories:
                    df.at[i + j, f"{model_key}_{cat}"] = None


    return df

# modelli
models_to_use = {
    "mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    # "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    # "gemma": "google/gemma-7b",
    # "gpt-j": "EleutherAI/gpt-j-6B"  # optional
}

In [8]:
# 🚀 Classifica con ogni modello
for model_key, model_id in models_to_use.items():
    print(f"\n📦 Loading model: {model_key}")
    pipe = load_model(model_id)

    # 📊 Classifica le battute con punteggi su tutte le categorie
    df = classify_jokes_with_scores(df, pipe, model_key=model_key, batch_size=32)

    # 📁 Salva CSV specifico per modello
    file_prefix = f"categoriz_{dataset_label}_{sample_label}_{model_key}"
    df.to_csv(f"/kaggle/working/{file_prefix}.csv", index=False)
    print(f"✅ Saved: {file_prefix}.csv")


📦 Loading model: mistral


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


📍 Model loaded on: cuda:0
🌀 [mistral] Batch 1


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


⚠️ JSON parsing error in batch 1, item 0: Extra data: line 10 column 9 (char 192)
📝 Model output:
You are a humor expert and researcher.
    
        For the following joke, rate from 0 to 10 how much it fits each of these categories based on their definitions.
        
        Categories:
        - "Edgy Content": Sex, taboos, death, shock
- "Cultural Reference": Politics, celebrities, pop culture
- "Wordplay": Puns, linguistic ambiguity
- "Absurdity": Illogical logic, surrealism, nonsense
- "Relatable": Everyday life, technology, relationships, self-irony
- "Offensive Humor": Insults, offensive, stereotypes, racism, sarcasm
        
        Joke:
        """What do all battered women have in common? They don't listen.
"""
        
        Respond ONLY in this exact JSON format:
        {
          "Edgy Content": 2,
          "Cultural Reference": 8,
          "Wordplay": 5,
          "Absurdity": 0,
          "Relatable": 3,
          "Insult Humor": 10 
        }

        The joke 

KeyboardInterrupt: 

In [ ]:
print(df.head(n=20))